# Predictive Maintenance Expert Assistant — RAG Chatbot

A Retrieval-Augmented Generation (RAG) chatbot specialized in **Remaining Useful Life (RUL) prediction** and predictive maintenance. Built with LlamaIndex, Groq for LLM inference, HuggingFace embeddings, and a Gradio chat UI.

**Pipeline overview:**
1. Load PDF/TXT source documents from Google Drive
2. Split them into chunks and embed them into a vector index
3. Wrap the index in a chat engine with a domain-specific system prompt
4. Serve the chatbot through a Gradio web interface

## 1. Connect to Google Drive

Source documents (PDF/TXT) and the persisted vector index both live on Google Drive so they survive Colab runtime resets.

In [1]:
from google.colab import drive

# Mount Google Drive so we can read the source PDFs/TXT files and,
# later, save/reload the vector index without re-uploading anything.
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install Dependencies

Colab comes with an older Python environment, so the required LlamaIndex modules are installed explicitly.

In [2]:
# Core RAG framework
!pip install -q llama-index-core
# LLM provider integration (Groq)
!pip install -q llama-index-llms-groq
# PDF / TXT file readers
!pip install -q llama-index-readers-file
# HuggingFace embedding models
!pip install -q llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Load API Keys

Keys are stored in Colab's Secrets manager (the key icon in the left sidebar), never hard-coded, so this notebook is safe to publish.

In [3]:
import os
from google.colab import userdata

# GROQ_API_KEY -> used by the LLM (Groq) for generating chat responses
# HF_TOKEN     -> used to download the HuggingFace embedding model
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 4. Initialize the LLM

Groq is used for fast, low-latency inference. `temperature=0.2` keeps answers grounded in the retrieved context while still allowing natural, well-explained phrasing (temperature=0 produced overly terse, copy-pasted answers during testing).

In [4]:
from llama_index.llms.groq import Groq

model = "openai/gpt-oss-120b"

# Low but non-zero temperature: stays faithful to the retrieved context
# (low hallucination risk) while still writing complete, natural sentences.
llm = Groq(model=model, temperature=0.2)

## 5. Load Source Documents

Reads every PDF/TXT file in the project folder. Each PDF page becomes one `Document` object, with the source filename and page number kept as metadata for later citation.

In [ ]:
from llama_index.core import SimpleDirectoryReader

# EDIT THIS: point it at your own Google Drive folder that contains your PDF/TXT source documents.
DATA_DIR = "/content/drive/MyDrive/YOUR_FOLDER_NAME"

# Reads all PDF/TXT files in the given folder and converts each page into a separate Document.
documents = SimpleDirectoryReader(DATA_DIR).load_data()

print(f"Loaded {len(documents)} document/page objects")

## 6. Split Documents into Chunks

Documents are split into smaller overlapping chunks so each one can be embedded and retrieved independently. `chunk_overlap` prevents a sentence from being cut in half at a chunk boundary.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# chunk_size=800 tokens keeps chunks large enough to preserve context;
# chunk_overlap=150 ensures a sentence at a chunk boundary still appears in full in at least one chunk.
text_splitter = SentenceSplitter(chunk_size=800, chunk_overlap=150)

docs = text_splitter.get_nodes_from_documents(documents)
print(f"Created {len(docs)} chunks")

## 7. Load the Embedding Model

`bge-base-en-v1.5` is a retrieval-optimized embedding model.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = "BAAI/bge-base-en-v1.5"
embeddings_folder = "/content/embedding_model/"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

## 8. Build the Vector Index (first run only)

Embeds every chunk and builds the vector index.

**Run this only once** — after it's persisted (next cell), future sessions can skip straight to Section 9.

In [ ]:
from llama_index.core import VectorStoreIndex

# Embeds all chunks and builds a searchable vector index.
# show_progress=True displays a progress bar since this step is slow on CPU.
vector_index = VectorStoreIndex(docs, embed_model=embeddings, show_progress=True)

In [ ]:
# EDIT THIS: choose where the built index should be saved on your Drive.
INDEX_DIR = "/content/drive/MyDrive/YOUR_FOLDER_NAME_index"

# Save the index to Google Drive so it survives Colab runtime resets and never needs to be recomputed from scratch again.
vector_index.storage_context.persist(persist_dir=INDEX_DIR)

## 9. Reload the Vector Index (subsequent runs)

**Alternative to Section 8.** If the index has already been built and persisted once, load it directly from Drive instead of re-embedding everything.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# EDIT THIS: same Drive path you used when you persisted the index (Section 8).
INDEX_DIR = "/content/drive/MyDrive/YOUR_FOLDER_NAME_index"

storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)

# Loads the previously saved index directly — no re-embedding needed.
# embed_model must match the one used when the index was built, since
# query vectors need to live in the same embedding space as the index.
vector_index = load_index_from_storage(storage_context, embed_model=embeddings)

## 10. Configure the Chat Engine



In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

# Keeps the last ~3000 tokens of conversation so the bot can handle follow-up questions.
memory = ChatMemoryBuffer.from_defaults(token_limit=3000)

system_prompt = (
    "You are a Predictive Maintenance Expert Assistant, specialized in Remaining Useful Life "
    "(RUL) prediction. You will be given several context passages, each possibly covering "
    "multiple sub-topics. Before answering, carefully scan EVERY passage in full — relevant "
    "information may be a single sentence embedded within a passage mostly about a different "
    "topic. Descriptive/qualitative explanations are valid — a formula is not required, but "
    "include one if it appears in the context. "
    "Write a complete, well-explained answer: define the concept clearly in your own words, "
    "use multiple sentences, and synthesize information from all relevant passages rather than "
    "just quoting a single isolated sentence. Aim for a short paragraph, not a one-line answer. "
    "If you have already given a clear answer to the question, do not contradict yourself "
    "afterward by claiming the information is missing — only flag missing information if a "
    "genuinely separate part of the question is truly unaddressed in the context. "
    "Do not bring in unrelated concepts (e.g. evaluation/scoring metrics) unless the question "
    "specifically asks about them."
)

chat_engine = vector_index.as_chat_engine(
    llm=llm,
    chat_mode="context",       # retrieves context for every message, no query rewriting
    memory=memory,
    system_prompt=system_prompt,
    similarity_top_k=6,        # number of chunks retrieved per question
)

# Start with a clean conversation history.
chat_engine.reset()

## 11. Test the Chat Engine

Quick sanity checks before wiring up the UI.

In [ ]:
r1 = chat_engine.chat("What is RUL?")
print(r1)

In [ ]:
r2 = chat_engine.chat("What is piecewise linear RUL and how to calculate it?")
print(r2)

## 12. Install Gradio

Colab ships with an older Gradio version (5.x); it must be explicitly upgraded to 6.x, which introduced breaking changes to the chat message format.

In [ ]:
# -U forces the upgrade even though an older Gradio is pre-installed on Colab.
!pip install -Uqqq gradio==6.3.0
import gradio as gr

## 13. Basic Chat Interface

The simplest working demo: wraps `chat_engine` in a Gradio `ChatInterface`. Note that this version shares one global memory across every visitor — see Section 14 for the fixed, multi-user-safe version.

In [ ]:
def rag_bot_chat_interface(message, history):
    # `history` is required by Gradio's function signature, but it is
    # unused here: chat_engine already tracks conversation history
    # internally via its own memory buffer.
    response = chat_engine.chat(message)
    return str(response)

In [ ]:
demo_chat_interface = gr.ChatInterface(
    fn=rag_bot_chat_interface,
    title="Predictive Maintenance Expert Assistant",
    description="Ask me questions about Remaining Useful Life (RUL) prediction and predictive maintenance.",
    examples=[
        "What is RUL?",
        "What is piecewise linear RUL and how to calculate it?",
        "How is RUL calculated in the C-MAPSS dataset?",
    ],
)

demo_chat_interface.launch(theme="soft", share=True)

## 14. Advanced Chat Interface (Per-User Memory + Adjustable Top-K)

The basic interface above shares a single `chat_engine` memory across every visitor — if two people use the app at the same time, they would see each other's conversation history, which is both confusing and a potential privacy issue.

This version fixes that: on every message, it resets the shared engine's memory and refills it with only the current browser session's own history (which Gradio already tracks per-user). It also exposes `similarity_top_k` as a live slider, so the number of retrieved chunks can be tuned from the UI.

In [ ]:
from llama_index.core.llms import ChatMessage


def custom_rag_bot_callback(message, history, top_k_value):
    # Let the UI control how many chunks are retrieved per question.
    chat_engine._retriever._similarity_top_k = int(top_k_value)

    # Reset the shared engine's memory, then refill it with ONLY this
    # user's own history (Gradio keeps `history` separate per session).
    # This prevents one user's conversation from leaking into another's.
    chat_engine.reset()
    bot_history = [
        ChatMessage(role=m["role"], content=m["content"][0]["text"])
        for m in history
    ]
    response = chat_engine.chat(message, chat_history=bot_history)

    return response.response


demo_chat_interface2 = gr.ChatInterface(
    fn=custom_rag_bot_callback,
    additional_inputs=[
        gr.Slider(minimum=1, maximum=5, step=1, value=2, label="Top K")
    ],
    title="Predictive Maintenance Expert Assistant",
    description="Ask me questions about Remaining Useful Life (RUL) prediction and predictive maintenance.",
    examples=[
        ["What is RUL?", 2],
        ["What is piecewise linear RUL and how to calculate it?", 2],
    ],
)

demo_chat_interface2.launch(theme="soft", share=True)